In [14]:
import torch
import torch.nn as nn
!pip install torchinfo
from torchinfo import summary

**1. Simple Neural Network (y_pred = sigmoid(w.x + b))**

In [23]:
class SimpleNN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        out = self.linear(x)
        out = self.sigmoid(out)
        return out

In [ ]:
input_ = torch.rand(10,5,dtype=torch.float32)
y = torch.randint(0,2,(input_.shape[0],),dtype=torch.float32)
input_, y

(tensor([[0.4707, 0.4460, 0.3486, 0.4919, 0.1908],
         [0.4069, 0.6509, 0.9890, 0.7088, 0.8466],
         [0.1705, 0.3100, 0.5128, 0.5636, 0.5657],
         [0.0325, 0.3805, 0.9079, 0.1862, 0.6594],
         [0.5207, 0.9798, 0.4449, 0.8178, 0.8207],
         [0.8311, 0.4330, 0.0568, 0.2968, 0.7086],
         [0.1490, 0.5285, 0.2788, 0.2859, 0.3578],
         [0.7069, 0.7162, 0.7237, 0.2875, 0.9428],
         [0.1256, 0.3515, 0.9354, 0.8724, 0.2581],
         [0.2340, 0.5246, 0.6592, 0.3458, 0.1944]]),
 tensor([1., 1., 0., 1., 0., 0., 1., 0., 0., 1.]))

In [28]:
learning_rate = 0.2
epochs = 20
loss_function = nn.BCELoss()

In [ ]:
model = SimpleNN(input_.shape[1])
optimizer = torch.optim.SGD(model.parameters(),lr = learning_rate)
for i in range(epochs):
    y_pred = model(input_)
    loss = loss_function(y_pred,y.view(-1,1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {i+1} - Loss: {loss}")

Epoch 1 - Loss: 0.6418163776397705
Epoch 2 - Loss: 0.6393764615058899
Epoch 3 - Loss: 0.6373001933097839
Epoch 4 - Loss: 0.6355092525482178
Epoch 5 - Loss: 0.6339421272277832
Epoch 6 - Loss: 0.6325508952140808
Epoch 7 - Loss: 0.6312979459762573
Epoch 8 - Loss: 0.6301542520523071
Epoch 9 - Loss: 0.6290966272354126
Epoch 10 - Loss: 0.6281075477600098


In [ ]:
print(f"Weight-gradient: {model.linear.weight}")
print(f"Bias-gradient: {model.linear.bias}")
summary(model)

Weight-gradient: Parameter containing:
tensor([[-0.4122, -0.1524,  0.4384, -0.3751, -0.4748]], requires_grad=True)
Bias-gradient: Parameter containing:
tensor([0.4049], requires_grad=True)


Layer (type:depth-idx)                   Param #
SimpleNN                                 --
├─Linear: 1-1                            6
├─Sigmoid: 1-2                           --
Total params: 6
Trainable params: 6
Non-trainable params: 0

**2. Deep Neural Network (y_pred = sigmoid(w.x + b))**

In [ ]:
class DeepNN(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(features,5),
            nn.ReLU(),
            nn.Linear(5,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        out = self.network(x)
        return out

In [ ]:
deep_model = DeepNN(input_.shape[1])
out_ = deep_model(input_)
print("Output of NN: ",output_)

Output of NN:  tensor([[0.6732],
        [0.6889],
        [0.6687],
        [0.6618],
        [0.6204],
        [0.5884],
        [0.6445],
        [0.6791],
        [0.6000],
        [0.6492]], grad_fn=<SigmoidBackward0>)


In [ ]:
summary(deep_model)

Layer (type:depth-idx)                   Param #
DeepNN                                   --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       30
│    └─ReLU: 2-2                         --
│    └─Linear: 2-3                       6
│    └─Sigmoid: 2-4                      --
Total params: 36
Trainable params: 36
Non-trainable params: 0

**3. Dataset Class & DatasetLoader Class**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
print(df.shape)
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
df.head(5)

(569, 33)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [7]:
X_train.shape, X_test.shape

((455, 30), (114, 30))

In [8]:
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

In [9]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [16]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [18]:
from torch.utils.data import Dataset, DataLoader

In [19]:
class MyDataset(Dataset):
    def __init__(self,x,y):
        self.x = x
        self.y = y

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self,index):
        return self.x[index], self.y[index]

In [20]:
train_data = MyDataset(X_train_tensor,y_train_tensor)
test_data = MyDataset(X_test_tensor, y_test_tensor)

In [21]:
train_loader = DataLoader(dataset=train_data,shuffle=True,batch_size=32)
test_loader = DataLoader(dataset=test_data,shuffle=True,batch_size=32)

In [24]:
model = SimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [29]:
for i in range(epochs):
    for feature,label in train_loader:
        y_pred = model(feature)
        loss = loss_function(y_pred,label.view(-1,1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch: {i+1}, Loss: {loss}")

Epoch: 1, Loss: 0.010833016596734524
Epoch: 2, Loss: 0.01962127909064293
Epoch: 3, Loss: 0.006514064036309719
Epoch: 4, Loss: 0.011953446082770824
Epoch: 5, Loss: 0.014733558520674706
Epoch: 6, Loss: 0.08097373694181442
Epoch: 7, Loss: 0.3820118010044098
Epoch: 8, Loss: 0.04446353763341904
Epoch: 9, Loss: 0.08630265295505524
Epoch: 10, Loss: 0.06743098050355911
Epoch: 11, Loss: 0.009699949994683266
Epoch: 12, Loss: 0.048686932772397995
Epoch: 13, Loss: 0.02924480102956295
Epoch: 14, Loss: 0.01504472829401493
Epoch: 15, Loss: 0.026839222759008408
Epoch: 16, Loss: 0.11313939094543457
Epoch: 17, Loss: 0.12999895215034485
Epoch: 18, Loss: 0.41794314980506897
Epoch: 19, Loss: 0.008870969526469707
Epoch: 20, Loss: 0.008724392391741276


In [30]:
model.eval()
accuracy_list = []
with torch.no_grad():
    for batch, label in test_loader:
        y_pred = model(batch)
        y_pred = (y_pred > 0.8).float()
        batch_accuracy = (y_pred.view(-1) == label).float().mean().item()
        accuracy_list.append(batch_accuracy)

accuracy = sum(accuracy_list)/len(accuracy_list)
print(f"Overall accuracy: {accuracy:.5f}")

Overall accuracy: 0.96267
